# 02 — Main Results

Primary specification: Does self-trading (total_self_trxn) causally affect total_gain?

Models:
1. Naive OLS (confounded baseline)
2. LinearDML (full panel, HistGBR nuisance)
3. LinearDML (within-trader demeaned — primary FE spec)
4. LinearDML (trader-level cross-section)
5. CausalForestDML (CATE heterogeneity)

In [39]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from src.data_loading import load_processed
from src.features import build_features, CORE_LAG_COLUMNS, get_country_dummies
from src.treatment_outcome import (
    compute_treatment, compute_outcome, demean_within_trader,
    collapse_to_trader_level, get_analysis_sample,
)
from src.models import naive_ols, fit_linear_dml, fit_causal_forest, prepare_model_arrays
from src.viz import (
    plot_margin_distributions, plot_causal_dag, plot_forest_comparison,
    plot_cate_distribution, plot_heterogeneity, plot_feature_importances,
    set_journal_style,
)

set_journal_style()
print('Imports OK')

Imports OK


## 1. Load Cleaned Data

In [40]:
parquet_path = '../data/processed/trader_panel.parquet' if os.path.exists('../data/processed/trader_panel.parquet') else 'data/processed/trader_panel.parquet'
df = load_processed(parquet_path)
print(f'Loaded: {len(df):,} rows, {df["trader_id"].nunique():,} traders')

2026-08-19 09:51:42,268 [INFO] src.data_loading: Loaded processed data: 1,222,680 rows from ../data/processed/trader_panel.parquet


Loaded: 1,222,680 rows, 26,580 traders


## 2. Feature Engineering

In [41]:
df = build_features(df, print_audit=True)

2026-08-19 09:51:42,354 [INFO] src.features: Starting feature engineering pipeline ...
2026-08-19 09:51:42,376 [INFO] src.features: Step 1: Computing lagged confounders
2026-08-19 09:51:42,682 [INFO] src.features:   Created 'learner_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,722 [INFO] src.features:   Created 'total_follower_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,757 [INFO] src.features:   Created 'guru_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,806 [INFO] src.features:   Created 'followed_count_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,823 [INFO] src.features:   Created 'gain_difference_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,873 [INFO] src.features:   Created 'greater_than_ten_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,911 [INFO] src.features:   Created 'total_trxn_l1': 26,580 NaN values (2.2% of rows)
2026-08-19 09:51:42,918 [INFO] src.features: Step 2: Lag verificatio


LAG VERIFICATION AUDIT TABLE
Verify: each _l1 value should equal the raw value from the PREVIOUS period for the same trader
 trader_id  period  learner_count  learner_count_l1  total_follower_count  total_follower_count_l1  guru_count  guru_count_l1
     23784      30              0               NaN                     0                      NaN           0            NaN
     23784      31              0               0.0                     0                      0.0           0            0.0
     23784      32              0               0.0                     0                      0.0           0            0.0
     23784      33              0               0.0                     0                      0.0           0            0.0
     23784      34              0               0.0                     0                      0.0           0            0.0
      5655      30              0               NaN                     0                      NaN           0         

2026-08-19 09:51:43,288 [WARNING] src.features: 48,929 rows have negative tenure (min_period_date < joining_date) — clamping to 0
2026-08-19 09:51:43,350 [INFO] src.features: Tenure distribution (valid):
count    1.222680e+06
mean     7.972680e+01
std      5.139616e+01
min      0.000000e+00
25%      4.714286e+01
50%      7.157143e+01
75%      1.021429e+02
max      3.240000e+02
Name: tenure_weeks, dtype: float64
2026-08-19 09:51:43,357 [INFO] src.features: Tenure skewness: 1.02
2026-08-19 09:51:43,414 [INFO] src.features: Step 4: Bucketing country
2026-08-19 09:51:44,055 [INFO] src.features: Country bucketing: 95 → 16 categories (top 15 + 'Other' + 'Unknown')
2026-08-19 09:51:44,107 [INFO] src.features: Feature engineering complete: 1,222,680 rows, 52 columns


## 3. Treatment & Outcome

In [42]:
df, treatment_stats = compute_treatment(df)
df = compute_outcome(df)

print('\nTreatment statistics:')
for k, v in treatment_stats.items():
    print(f'  {k}: {v}')

2026-08-19 09:51:44,240 [INFO] src.treatment_outcome: Active weeks (total_trxn > 0): 81,266 (6.6%) — Inactive: 1,141,414
2026-08-19 09:51:44,304 [INFO] src.treatment_outcome: Treatment (total_self_trxn) full panel: mean=0.148, median=0.0, zero share=98.2%
2026-08-19 09:51:44,679 [INFO] src.treatment_outcome: Winsorized total_gain at [1%, 99%]: bounds = [-100.91, 79.41]



Treatment statistics:
  total_rows: 1222680
  active_weeks: 81266
  inactive_weeks: 1141414
  active_fraction: 0.06646546929695424
  treatment_mean: 0.14807635685543233
  treatment_median: 0.0
  treatment_std: 2.1094765718009474
  treatment_zero_share: 0.9815659044067131
  ratio_mean_active: 0.211141972153676
  ratio_median_active: 0.0
  ratio_zero_share_active: 0.7226515393891665
  ratio_one_share_active: 0.17232298870376295


In [43]:
# Diagnostic: Extensive vs intensive margin
plot_margin_distributions(df, variable='total_self_trxn', figdir='../figures')

2026-08-19 09:51:45,341 [INFO] src.viz: Saved: ../figures\diag_margins_total_self_trxn.png, ../figures\diag_margins_total_self_trxn.pdf


'../figures\\diag_margins_total_self_trxn.png'

## 4. Analysis Sample Construction

In [44]:
confounder_lag_cols = [f'{c}_l1' for c in CORE_LAG_COLUMNS]
heterogeneity_cols = ['learner_count_l1', 'total_follower_count_l1', 'tenure_weeks']

df_analysis, sample_counts = get_analysis_sample(
    df, treatment_col='total_self_trxn',
    confounder_lag_cols=confounder_lag_cols,
    require_tenure=True,
)

print(f'\nAnalysis sample: {len(df_analysis):,} rows, {df_analysis["trader_id"].nunique():,} traders')
print(f'Sample construction: {sample_counts}')

2026-08-19 09:51:45,363 [INFO] src.treatment_outcome:   Step 1 (treatment not null): 1,222,680 → 1,222,680 (dropped 0)
2026-08-19 09:51:45,371 [INFO] src.treatment_outcome:   Step 3 (lag learner_count_l1 not null): dropped 26,580
2026-08-19 09:51:45,397 [INFO] src.treatment_outcome:   Step 4 (tenure not null): dropped 0
2026-08-19 09:51:45,843 [INFO] src.treatment_outcome: Sample construction: 1,222,680 → 1,196,100 (26,580 traders)



Analysis sample: 1,196,100 rows, 26,580 traders
Sample construction: {'total': 1222680, 'after_treatment_filter': 1222680, 'dropped_null_treatment': 0, 'after_outcome_filter': 1222680, 'dropped_null_outcome': 0, 'after_lag_filter': 1196100, 'dropped_missing_lags': 26580, 'after_tenure_filter': 1196100, 'dropped_missing_tenure': 0, 'final_rows': 1196100, 'final_traders': 26580}


In [45]:
# Country dummies
if 'user_country_name_bucketed' in df_analysis.columns:
    df_analysis, country_dummy_cols = get_country_dummies(df_analysis)
    print(f'Country dummies: {len(country_dummy_cols)} columns')
else:
    country_dummy_cols = []

Country dummies: 15 columns


## 5. Causal DAG

In [46]:
plot_causal_dag(figdir='../figures')

2026-08-19 09:51:46,700 [INFO] src.viz: Saved: ../figures\fig2_causal_dag.png, ../figures\fig2_causal_dag.pdf


'../figures\\fig2_causal_dag.png'

## 6. Within-Trader Demeaning

In [47]:
demean_cols = ['total_self_trxn', 'total_gain_w'] + confounder_lag_cols
df_analysis = demean_within_trader(df_analysis, demean_cols)
print('Demeaning complete.')

2026-08-19 09:51:47,284 [INFO] src.treatment_outcome:   Demeaned 'total_self_trxn' → 'total_self_trxn_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,295 [INFO] src.treatment_outcome:   Demeaned 'total_gain_w' → 'total_gain_w_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,305 [INFO] src.treatment_outcome:   Demeaned 'learner_count_l1' → 'learner_count_l1_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,314 [INFO] src.treatment_outcome:   Demeaned 'total_follower_count_l1' → 'total_follower_count_l1_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,324 [INFO] src.treatment_outcome:   Demeaned 'guru_count_l1' → 'guru_count_l1_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,333 [INFO] src.treatment_outcome:   Demeaned 'followed_count_l1' → 'followed_count_l1_demeaned' (mean of demeaned: 0.000000)
2026-08-19 09:51:47,342 [INFO] src.treatment_outcome:   Demeaned 'gain_difference_l1' → 'gain_difference_l1_demeaned' (mean of demeaned: 0.000

Demeaning complete.


## 7. Model Fitting

### 7a. Naive OLS

In [48]:
ols_result = naive_ols(
    Y=df_analysis['total_gain_w'].values,
    T=df_analysis['total_self_trxn'].values,
)
print(f"Naive OLS: β={ols_result['coef']:.4f}, 95% CI=[{ols_result['ci_lower']:.4f}, {ols_result['ci_upper']:.4f}]")

2026-08-19 09:51:47,567 [INFO] src.models: Naive OLS: coef=-0.5605, SE=0.0444, 95% CI=[-0.6475, -0.4735], p=1.408e-36, N=1,196,100


Naive OLS: β=-0.5605, 95% CI=[-0.6475, -0.4735]


### 7b. LinearDML (Full Panel)

In [49]:
all_confounder_cols = confounder_lag_cols + ['tenure_weeks']
all_confounder_cols_wc = all_confounder_cols + country_dummy_cols

arrays = prepare_model_arrays(
    df_analysis,
    treatment_col='total_self_trxn',
    outcome_col='total_gain_w',
    confounder_cols=all_confounder_cols_wc,
)

_, dml_full = fit_linear_dml(
    Y=arrays['Y'], T=arrays['T'], W=arrays['W'],
    nuisance_learner='histgbr',
    cache_name='dml_full_histgbr',
)
print(f"LinearDML (full): ATE={dml_full['ate']:.4f}, CI=[{dml_full['ate_ci_lower']:.4f}, {dml_full['ate_ci_upper']:.4f}]")

2026-08-19 09:51:47,753 [INFO] src.models: Model arrays: Y((1196100,)), T((1196100,)), W((1196100, 23)), X(None)
2026-08-19 09:51:47,769 [INFO] src.models: Loading cached LinearDML from data/processed\dml_full_histgbr.pkl


LinearDML (full): ATE=-0.5980, CI=[-0.6987, -0.4974]


### 7c. LinearDML (Within-Trader Demeaned) — Primary FE Spec

In [50]:
demeaned_conf = [f'{c}_demeaned' for c in confounder_lag_cols if f'{c}_demeaned' in df_analysis.columns]

arrays_dm = prepare_model_arrays(
    df_analysis,
    treatment_col='total_self_trxn_demeaned',
    outcome_col='total_gain_w_demeaned',
    confounder_cols=demeaned_conf,
)

_, dml_demeaned = fit_linear_dml(
    Y=arrays_dm['Y'], T=arrays_dm['T'], W=arrays_dm['W'],
    nuisance_learner='histgbr',
    cache_name='dml_demeaned_histgbr',
)
dml_demeaned['model_name'] = 'LinearDML (within-trader demeaned)'
print(f"LinearDML (demeaned): ATE={dml_demeaned['ate']:.4f}, CI=[{dml_demeaned['ate_ci_lower']:.4f}, {dml_demeaned['ate_ci_upper']:.4f}]")

2026-08-19 09:51:47,863 [INFO] src.models: Model arrays: Y((1196100,)), T((1196100,)), W((1196100, 7)), X(None)
2026-08-19 09:51:47,869 [INFO] src.models: Loading cached LinearDML from data/processed\dml_demeaned_histgbr.pkl


LinearDML (demeaned): ATE=-0.5884, CI=[-0.6870, -0.4898]


### 7d. LinearDML (Trader-Level Cross-Section)

In [51]:
df_trader = collapse_to_trader_level(
    df_analysis, confounder_cols=all_confounder_cols,
    treatment_col='total_self_trxn', outcome_col='total_gain_w',
)
valid_cols_tl = [c for c in all_confounder_cols if c in df_trader.columns]
df_trader_clean = df_trader.dropna(subset=valid_cols_tl + ['total_self_trxn', 'total_gain_w'])

arrays_tl = prepare_model_arrays(
    df_trader_clean,
    treatment_col='total_self_trxn', outcome_col='total_gain_w',
    confounder_cols=valid_cols_tl,
)

_, dml_trader = fit_linear_dml(
    Y=arrays_tl['Y'], T=arrays_tl['T'], W=arrays_tl['W'],
    nuisance_learner='histgbr',
    cache_name='dml_trader_level',
)
dml_trader['model_name'] = 'LinearDML (trader-level cross-section)'
print(f"LinearDML (trader): ATE={dml_trader['ate']:.4f}, CI=[{dml_trader['ate_ci_lower']:.4f}, {dml_trader['ate_ci_upper']:.4f}]")

2026-08-19 09:51:49,178 [INFO] src.treatment_outcome: Collapsed panel to trader level: 26,580 traders, 12 columns
2026-08-19 09:51:49,233 [INFO] src.models: Model arrays: Y((26580,)), T((26580,)), W((26580, 8)), X(None)
2026-08-19 09:51:49,233 [INFO] src.models: Loading cached LinearDML from data/processed\dml_trader_level.pkl


LinearDML (trader): ATE=-0.3774, CI=[-0.5113, -0.2434]


### 7e. Forest Plot: All Estimates

In [52]:
plot_forest_comparison(
    [ols_result, dml_full, dml_demeaned, dml_trader],
    figdir='../figures',
)

2026-08-19 09:51:49,518 [INFO] src.viz: Saved: ../figures\fig3_forest_comparison.png, ../figures\fig3_forest_comparison.pdf


'../figures\\fig3_forest_comparison.png'

### 7f. CausalForestDML (CATE)

In [53]:
het_cols = [c for c in heterogeneity_cols if c in df_analysis.columns]
w_cols_forest = [c for c in all_confounder_cols_wc if c not in het_cols]

arrays_cf = prepare_model_arrays(
    df_analysis,
    treatment_col='total_self_trxn', outcome_col='total_gain_w',
    confounder_cols=w_cols_forest, heterogeneity_cols=het_cols,
)

_, cf_results, cate = fit_causal_forest(
    Y=arrays_cf['Y'], T=arrays_cf['T'],
    X=arrays_cf['X'], W=arrays_cf['W'],
    max_samples=200000,
    cache_name='causal_forest',
)

print(f"CausalForest ATE: {cf_results['ate_from_cate']:.4f}")
print(f"CATE: mean={cf_results['cate_mean']:.4f}, median={cf_results['cate_median']:.4f}")
print(f"Share positive: {cf_results['share_positive']:.1%}")

2026-08-19 09:51:49,707 [INFO] src.models: Model arrays: Y((1196100,)), T((1196100,)), W((1196100, 20)), X((1196100, 3))
2026-08-19 09:51:49,719 [INFO] src.models: Loading cached CausalForestDML from data/processed\causal_forest.pkl


CausalForest ATE: -1.4749
CATE: mean=-1.4749, median=-0.9733
Share positive: 35.4%


In [54]:
# CATE distribution
plot_cate_distribution(cate, figdir='../figures')

2026-08-19 09:51:58,591 [INFO] src.viz: Saved: ../figures\fig4_cate_distribution.png, ../figures\fig4_cate_distribution.pdf


'../figures\\fig4_cate_distribution.png'

In [55]:
# Heterogeneity by tenure and audience
plot_heterogeneity(
    cate=cate,
    tenure=df_analysis['tenure_weeks'].values,
    audience=df_analysis['learner_count_l1'].values,
    figdir='../figures',
)

D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\viz.py:467: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).groupby("bin")["cate"]
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\viz.py:467: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ).groupby("bin")["cate"]
2026-08-19 09:51:59,233 [INFO] src.viz: Saved: ../figures\fig5_heterogeneity.png, ../figures\fig5_heterogeneity.pdf


'../figures\\fig5_heterogeneity.png'

In [56]:
# Feature importances
if 'feature_importances' in cf_results:
    plot_feature_importances(
        np.array(cf_results['feature_importances']),
        het_cols,
        figdir='../figures',
    )

2026-08-19 09:51:59,841 [INFO] src.viz: Saved: ../figures\fig6_feature_importances.png, ../figures\fig6_feature_importances.pdf


## 8. Results Summary

In [57]:
print('\n' + '='*70)
print('RESULTS SUMMARY')
print('='*70)
for label, r in [('Naive OLS', ols_result), ('DML (full panel)', dml_full),
                  ('DML (demeaned/FE)', dml_demeaned), ('DML (trader-level)', dml_trader)]:
    coef = r.get('ate', r.get('coef', None))
    ci_lo = r.get('ate_ci_lower', r.get('ci_lower', None))
    ci_hi = r.get('ate_ci_upper', r.get('ci_upper', None))
    n = r.get('n', '')
    print(f'  {label:30s}  β={coef:.4f}  [{ci_lo:.4f}, {ci_hi:.4f}]  N={n:,}')
print(f'  {"CausalForest (ATE from CATE)":30s}  β={cf_results["ate_from_cate"]:.4f}  share>0={cf_results["share_positive"]:.1%}')
print('='*70)


RESULTS SUMMARY
  Naive OLS                       β=-0.5605  [-0.6475, -0.4735]  N=1,196,100
  DML (full panel)                β=-0.5980  [-0.6987, -0.4974]  N=1,196,100
  DML (demeaned/FE)               β=-0.5884  [-0.6870, -0.4898]  N=1,196,100
  DML (trader-level)              β=-0.3774  [-0.5113, -0.2434]  N=26,580
  CausalForest (ATE from CATE)    β=-1.4749  share>0=35.4%
